# A3. Backbone

Does a transformer beat the U-Net on DLA?

See `docs/plans/experiments-plan.md`, entry A3. Results: `artifacts/results/dla_test/`.

## Run

From the repo root. `uv run erosion-sim list` shows what is already trained and evaluated.

**1. Data.** Skip any that already exist.

Local:

```sh
uv run erosion-sim dla-generate dla_master
```

Cluster:

```sh
sbatch slurm/dla_generate.sbatch dla_master
```

**2. Train**, in this order. Each resumes if stopped; a finished model is skipped.

Local:

```sh
uv run erosion-sim train dla_base
uv run erosion-sim train dla_dit
```

Cluster:

```sh
sbatch slurm/train.sbatch dla_base
sbatch slurm/train.sbatch dla_dit
```

**3. Evaluate**, once training has finished. It refuses to replace existing results.

Local:

```sh
uv run erosion-sim evaluate dla_test dla_base
uv run erosion-sim evaluate dla_test dla_dit
```

Cluster:

```sh
sbatch slurm/evaluate.sbatch dla_test dla_base
sbatch slurm/evaluate.sbatch dla_test dla_dit
```

Then run the cells below.

How to read the tables and plots: [`docs/guides/dla.md`](../../docs/guides/dla.md).

In [ ]:
import json

from erosion_sim.dla import plots
from erosion_sim.evaluation.results import load
from erosion_sim.paths import model_dir
from erosion_sim.training.train import plot_history

EVALUATION = "dla_test"  # configs/evaluations/dla_test.yaml
PREDICTORS = ["dla_base", "dla_dit"]

results = load(EVALUATION, *PREDICTORS)
{p: {k: m[k] for k in ("time", "code", "slurm_job")} for p, m in results.manifest["predictors"].items()}

## Training

Compare time per step too.

In [ ]:
for name in PREDICTORS:
    print(name, model_dir(name))
    plot_history(json.loads((model_dir(name) / "history.json").read_text()))
for name in PREDICTORS:
    done = json.loads((model_dir(name) / "done.json").read_text())
    print(name, f"{done['seconds'] / done['step'] * 1000:.1f} ms per step")

In [ ]:
plots.table(results)

In [ ]:
for metric in plots.METRIC_LABELS:
    plots.plot_metric(results, metric)

In [ ]:
plots.plot_examples(results)

## Conclusion

_Written after reading the results above._

A5 and A6 use `dla_base`. If another model wins A2 to A4, change `PREDICTORS` there and the commands above.